# Ollama qwen2.5:7b Demo

This notebook shows two ways to talk to a local Ollama model:

- Python API via the `ollama` package
- Raw HTTP requests via `requests`

Default model: `qwen2.5:7b`

In [ ]:
%pip install -q ollama requests

In [ ]:
MODEL = "qwen2.5:7b"
BASE_URL = "http://127.0.0.1:11434"

print(f"Model: {MODEL}")
print(f"Base URL: {BASE_URL}")

In [ ]:
from ollama import Client

client = Client(host=BASE_URL)

def ask_ollama_api(prompt: str, model: str = MODEL) -> str:
    response = client.chat(
        model=model,
        messages=[
            {"role": "system", "content": "You are a concise and friendly assistant."},
            {"role": "user", "content": prompt},
        ],
    )
    return response["message"]["content"]


api_reply = ask_ollama_api("Reply with exactly: hello from ollama api")
print(api_reply)

In [ ]:
import requests

def ask_ollama_http(prompt: str, model: str = MODEL) -> str:
    url = f"{BASE_URL}/api/chat"
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": "You are a concise and friendly assistant."},
            {"role": "user", "content": prompt},
        ],
        "stream": False,
    }
    resp = requests.post(url, json=payload, timeout=120)
    resp.raise_for_status()
    data = resp.json()
    return data["message"]["content"]


http_reply = ask_ollama_http("Reply with exactly: hello from ollama http")
print(http_reply)

## Streaming versions

The next two cells show streaming output for both the Python API and raw HTTP.

In [ ]:
def ask_ollama_api_stream(prompt: str, model: str = MODEL) -> str:
    stream = client.chat(
        model=model,
        messages=[
            {"role": "system", "content": "You are a concise and friendly assistant."},
            {"role": "user", "content": prompt},
        ],
        stream=True,
    )

    chunks = []
    for part in stream:
        delta = part.get("message", {}).get("content", "") if isinstance(part, dict) else getattr(getattr(part, "message", None), "content", "")
        if delta:
            print(delta, end="", flush=True)
            chunks.append(delta)

    print()
    return "".join(chunks)


# Uncomment to test the streaming API version.
# api_stream_reply = ask_ollama_api_stream("Count from 1 to 5 in one line.")

In [ ]:
import json

def ask_ollama_http_stream(prompt: str, model: str = MODEL) -> str:
    url = f"{BASE_URL}/api/chat"
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": "You are a concise and friendly assistant."},
            {"role": "user", "content": prompt},
        ],
        "stream": True,
    }

    chunks = []
    with requests.post(url, json=payload, stream=True, timeout=120) as resp:
        resp.raise_for_status()
        for line in resp.iter_lines(decode_unicode=True):
            if not line:
                continue
            data = json.loads(line)
            delta = data.get("message", {}).get("content", "")
            if delta:
                print(delta, end="", flush=True)
                chunks.append(delta)
            if data.get("done"):
                break

    print()
    return "".join(chunks)


# Uncomment to test the streaming HTTP version.
# http_stream_reply = ask_ollama_http_stream("Count from 1 to 5 in one line.")

In [ ]:
print("Done")